In [1]:
import pandas as pd
import numpy as np 
from pathlib import Path
from os.path import join, getsize
import kaldiio # for use with kaldi type format
import pyarrow as pa
from pyarrow import csv
import json 

In [2]:
path = '/om2/data/public/GigaSpeech/'

split = 'XL'
# split = 'DEV'
audio_path = Path(join(path, 'audio'))
samp_rate = 16000
manifest_path = Path(f"{path}/data/{split}.csv")

In [3]:
manifest = pd.read_csv(manifest_path, sep='\t',
                               usecols=['wav_filename', 'speaker', 'transcript']) 

In [4]:
manifest = manifest[~manifest.isna().any(axis=1)] # toss nan file here

In [5]:
manifest.isna().any()

wav_filename    False
transcript      False
speaker         False
dtype: bool

In [6]:
segments = pd.read_csv(Path(path,'data', 'segments'), header=None, 
                       names=['wav_filename', 'speaker', 'start', 'end'], sep='\t')

In [7]:
# wavscp_k = kaldiio.load_scp(Path(path,'data','wav.scp').as_posix())
wavscp = pd.read_csv(Path(path,'data','wav.scp'), header=None,
                                 names=['speaker', 'wav_path'], sep='\t',
                                 )

In [8]:
new_man = pd.merge(manifest,segments, on=['wav_filename', 'speaker'])

In [9]:
new_man = pd.merge(new_man, wavscp, on='speaker')

In [10]:
new_man.isna().any()

wav_filename    False
transcript      False
speaker         False
start           False
end             False
wav_path        False
dtype: bool

In [11]:
new_man.head()

,wav_filename,transcript,speaker,start,end,wav_path
0,POD0000000001_S0000008,DOUGLAS MCGRAY IS GOING TO BE OUR GUIDE YOU WA...,POD0000000001,159.00,167.52,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
1,POD0000000001_S0000010,AH ITALIAN AND AND THEN YOU HAVE A CONVERSATIO...,POD0000000001,195.46,208.22,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
2,POD0000000001_S0000011,DOUG MCGRAY IS A JOURNALIST MY NAME IS DOUGLAS...,POD0000000001,209.54,229.52,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
3,POD0000000001_S0000012,AH YOU LEAVE WITH CASH MINUS A THREE PERCENT F...,POD0000000001,229.70,232.88,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
4,POD0000000001_S0000013,AND PAYDAY LENDING IS WHEN YOU DON'T HAVE ANY ...,POD0000000001,232.88,251.94,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...


In [12]:
new_man.sort_values(by="transcript", key=lambda x: x.str.len(), inplace=True)

In [14]:
new_man.head()

,wav_filename,transcript,speaker,start,end,wav_path
7413942,AUD0000000304_S0000880,O,AUD0000000304,3931.99,3932.77,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
4022145,YOU0000009259_S0000358,I,YOU0000009259,1476.61,1477.39,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
3723129,YOU0000007664_S0000126,O,YOU0000007664,2044.05,2044.80,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
3723128,YOU0000007664_S0000125,M,YOU0000007664,2042.22,2043.03,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
3723127,YOU0000007664_S0000124,S,YOU0000007664,2040.33,2041.26,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...


In [11]:
# parse_opts = csv.ParseOptions(delimiter='\t')
# table_manifest = csv.read_csv(manifest_path, parse_options=parse_opts)

In [15]:
out_path = Path(path,'data','XL_munged.csv')

new_man.to_csv(out_path, index=False)

In [11]:
# train_dict = new_man.to_dict(orient='records')

In [16]:
# out_path = Path(path,'data','XL_munged.json')

# with open(out_path, 'w') as fp:
#     json.dump(train_dict, fp)

In [17]:
split = 'DEV'
manifest_path = Path(f"{path}/data/{split}.csv")
manifest = pd.read_csv(manifest_path, sep='\t',
                               usecols=['wav_filename', 'speaker', 'transcript']) 

In [18]:
dev_man = pd.merge(manifest,segments, on=['wav_filename', 'speaker'])

In [19]:
dev_man = pd.merge(dev_man, wavscp, on='speaker')

In [20]:
dev_dict = dev_man.to_dict(orient='records')

In [23]:
dev_man.loc[0,'wav_path']

'/rdma/vast-rdma/om3/data/public/GigaSpeech/audio/podcast/P0000/POD1000000004.wav'

In [24]:
out_path = Path(path,'data','DEV_munged.json')

with open(out_path, 'w') as fp:
    json.dump(dev_dict, fp)

In [28]:
out_path = Path(path,'data','DEV_munged.csv')

dev_man.to_csv(out_path, index=False)